In [ ]:
import keras
import catppuccin
import matplotlib
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

matplotlib.style.use(["dark_background", catppuccin.PALETTE.mocha.identifier])


In [ ]:
import kagglehub

path = kagglehub.dataset_download("puneet6060/intel-image-classification")
path


In [ ]:
train_data = keras.utils.image_dataset_from_directory(
    directory=path + "/seg_train/seg_train",
    labels="inferred",
    batch_size=32,
    image_size=(224, 224),
    shuffle=True,
)


In [ ]:
val_data = keras.utils.image_dataset_from_directory(
    directory=path + "/seg_test/seg_test",
    labels="inferred",
    batch_size=32,
    image_size=(224, 224),
    shuffle=False,
)


In [ ]:
n = len(train_data.class_names)
n


In [ ]:
from keras.applications.resnet_v2 import ResNet50V2, preprocess_input

base_model = ResNet50V2(
    include_top=True,
    weights="imagenet",
    input_shape=(224, 224, 3),
)
base_model.trainable = False

In [ ]:
from keras.layers import Dense, Input, Lambda

model = keras.Sequential(
    [
        Input(shape=(224, 224, 3)),
        keras.layers.Lambda(preprocess_input),
        base_model,
        Dense(128, activation="relu"),
        Dense(64, activation="relu"),
        Dense(n, activation="softmax"),
    ]
)
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

In [ ]:
history = model.fit(train_data, epochs=5, validation_data=val_data)


In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]


plt.figure(figsize=(20, 10))
plt.plot(acc, label="acc")
plt.plot(val_acc, label="val_acc")
plt.legend()
plt.plot()
